In [1]:
from  pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import os


In [2]:
spark = SparkSession.builder \
    .appName('Office Products Review Cleaning Phase 2') \
    .master('local[4]') \
    .config('spark.driver.memory', '16g') \
    .config('spark.driver.maxResultSize', '10g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer') \
    .config('spark.sql.parquet.compression.codec', 'gzip') \
    .config('spark.default.parallelism', '4') \
    .getOrCreate()


your 131072x1 screen size is bogus. expect trouble
26/04/02 18:11:54 WARN Utils: Your hostname, kien resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/02 18:11:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 18:11:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.sparkContext.setLogLevel('WARN')

df = spark.read.parquet("../data/processed/Office_Products_Cleaned.parquet")
initial_sentence_count = df.count()
initial_review_count = df.select('review_id').distinct().count()
df.show(5, truncate=50)

+-----------+---------+-----------+--------------------------------------------------+------+
|parent_asin|review_id|sentence_id|                                     sentence_text|rating|
+-----------+---------+-----------+--------------------------------------------------+------+
| B00PT7YRBM|       28|          1|ASINB00PT7YRBM The Ultimate Preschool Curriculu...|   5.0|
| B00PT7YRBM|       28|          2|Beautiful and complex curriculum for preschoole...|   5.0|
| B00PT7YRBM|       28|          3|    Cant wait to start using it in my home daycare|   5.0|
| B00PT7YRBM|       28|          4|                                         Thank you|   5.0|
| B0BW7SXY8B|       33|          1|Only thing I didnt like was the notepad which c...|   5.0|
+-----------+---------+-----------+--------------------------------------------------+------+
only showing top 5 rows



In [ ]:
CONTRACTIONS = {
    "won't": "will not", "can't": "can not", "shan't": "shall not",
    "n't": " not", "'re": " are", "'ve": " have",
    "'ll": " will", "'d": " would", "'m": " am",
    "let's": "let us", "it's": "it is", "that's": "that is",
    "what's": "what is", "there's": "there is", "here's": "here is",
    "who's": "who is", "how's": "how is", "where's": "where is",
    "he's": "he is", "she's": "she is",
    "i'm": "i am", "you're": "you are", "we're": "we are", "they're": "they are",
    "i've": "i have", "you've": "you have", "we've": "we have", "they've": "they have",
    "i'll": "i will", "you'll": "you will", "he'll": "he will", "she'll": "she will",
    "we'll": "we will", "they'll": "they will",
    "i'd": "i would", "you'd": "you would", "he'd": "he would", "she'd": "she would",
    "we'd": "we would", "they'd": "they would",
    "isn't": "is not", "aren't": "are not", "wasn't": "was not", "weren't": "were not",
    "hasn't": "has not", "haven't": "have not", "hadn't": "had not",
    "doesn't": "does not", "don't": "do not", "didn't": "did not",
    "wouldn't": "would not", "shouldn't": "should not", "couldn't": "could not",
    "mustn't": "must not", "needn't": "need not",
}

bc_contractions = spark.sparkContext.broadcast(CONTRACTIONS)

@udf(StringType())
def normalize_text(text):
    if not text:
        return text
    import re
    text = text.lower()
    contractions = bc_contractions.value
    for k, v in sorted(contractions.items(), key=lambda x: -len(x[0])):
        text = text.replace(k, v)

    text = re.sub(r"[^a-z0-9\s']", ' ', text)
    text = re.sub(r"\s+", ' ', text).strip()
    return text

df = df.withColumn('sentence_text', normalize_text('sentence_text'))
df = df.filter(
    (col('sentence_text').isNotNull()) &
    (trim('sentence_text') != '')
)
after_normalize = df.cache().count()
print(f'Chuẩn hóa text. Còn {after_normalize:,} câu')
df.show(5, truncate=50)

Chuẩn hóa text. Còn {after_normalize:,} câu
+-----------+---------+-----------+--------------------------------------------------+------+
|parent_asin|review_id|sentence_id|                                     sentence_text|rating|
+-----------+---------+-----------+--------------------------------------------------+------+
| B00PT7YRBM|       28|          1|asinb00pt7yrbm the ultimate preschool curriculu...|   5.0|
| B00PT7YRBM|       28|          2|beautiful and complex curriculum for preschoole...|   5.0|
| B00PT7YRBM|       28|          3|    cant wait to start using it in my home daycare|   5.0|
| B00PT7YRBM|       28|          4|                                         thank you|   5.0|
| B0BW7SXY8B|       33|          1|only thing i didnt like was the notepad which c...|   5.0|
+-----------+---------+-----------+--------------------------------------------------+------+
only showing top 5 rows



In [5]:
GENERIC_NOUNS = {
    'thing', 'things', 'stuff', 'item', 'items', 'product', 'products',
    'object', 'objects', 'part', 'parts', 'piece', 'pieces',
    'kind', 'kinds', 'type', 'types', 'sort', 'sorts',
    'one', 'ones', 'something', 'anything', 'everything', 'nothing',
    'someone', 'anyone', 'everyone', 'nobody',
    'aspect', 'issue', 'issues', 'lot', 'bit', 'matter', 'way',
    'case', 'point', 'side', 'version',
    'time', 'times', 'day', 'days', 'week', 'weeks',
    'month', 'months', 'year', 'years',
    'problem', 'problems', 'reason', 'reasons', 'result', 'results',
    'experience', 'review', 'reviews', 'rating', 'ratings', 'star', 'stars'
}

DOMAIN_NOISE = {
    'amazon', 'seller', 'sellers', 'shipping', 'delivery',
    'shipment', 'shipments', 'package', 'packages', 'packaging',
    'box', 'boxes', 'wrapper', 'wrappers',
    'return', 'returns', 'refund', 'refunds',
    'replacement', 'replacements'
}

bc_generic = spark.sparkContext.broadcast(GENERIC_NOUNS)
bc_noise = spark.sparkContext.broadcast(DOMAIN_NOISE)

@udf(StringType())
def tag_special_words(text):
    if not text:
        return text
    generic = bc_generic.value
    noise = bc_noise.value
    words = text.split()
    result = []
    for w in words:
        w_clean = w.strip("'")
        if w_clean in generic:
            result.append('[GENERIC_NOUN]')
        elif w_clean in noise:
            result.append('[DOMAIN_NOISE]')
        else:
            result.append(w)
    return ' '.join(result)

df.unpersist()
df = df.withColumn('sentence_text', tag_special_words('sentence_text'))
print('Đánh dấu GENERIC_NOUN và DOMAIN_NOISE')
df.show(5, truncate=50)

Đánh dấu GENERIC_NOUN và DOMAIN_NOISE


+-----------+---------+-----------+--------------------------------------------------+------+
|parent_asin|review_id|sentence_id|                                     sentence_text|rating|
+-----------+---------+-----------+--------------------------------------------------+------+
| B00PT7YRBM|       28|          1|asinb00pt7yrbm the ultimate preschool curriculu...|   5.0|
| B00PT7YRBM|       28|          2|beautiful and complex curriculum for preschoole...|   5.0|
| B00PT7YRBM|       28|          3|    cant wait to start using it in my home daycare|   5.0|
| B00PT7YRBM|       28|          4|                                         thank you|   5.0|
| B0BW7SXY8B|       33|          1|only [GENERIC_NOUN] i didnt like was the notepa...|   5.0|
+-----------+---------+-----------+--------------------------------------------------+------+
only showing top 5 rows



In [6]:
from pyspark.sql.functions import col, split, size, sum

result = df.select(
    sum(size(split(col('sentence_text'), '\\[GENERIC_NOUN\\]')) - 1).alias("generic_count"),
    sum(size(split(col('sentence_text'), '\\[DOMAIN_NOISE\\]')) - 1).alias("noise_count")
).first()

generic_count = result["generic_count"] or 0
noise_count = result["noise_count"] or 0

print(f'Số lượng GENERIC_NOUN tags: {generic_count:,}')
print(f'Số lượng DOMAIN_NOISE tags: {noise_count:,}')

Số lượng GENERIC_NOUN tags: 15,963,111
Số lượng DOMAIN_NOISE tags: 2,208,100


In [7]:
from pyspark.sql.functions import col, length, size, split, when, sum


df = df.withColumn("wc", size(split(col("sentence_text"), "\\s+")))


result = df.select(
    sum(
        when(
            (length(col('sentence_text')) < 5) | (col("wc") < 2),
            1
        ).otherwise(0)
    ).alias("short_count"),
    sum(
        when(
            (length(col('sentence_text')) >= 5) & (col("wc") >= 2),
            1
        ).otherwise(0)
    ).alias("valid_count")
).first()

short_count = result["short_count"]
after_short = result["valid_count"]


df = df.filter(
    (length(col('sentence_text')) >= 5) & (col("wc") >= 2)
).drop("wc")

print(f'Xóa {short_count:,} câu quá ngắn. Còn {after_short:,} câu')

Xóa 951,579 câu quá ngắn. Còn 40,093,448 câu


In [8]:
PRONOUNS = {'i', 'me', 'my', 'mine', 'myself',
            'you', 'your', 'yours', 'yourself',
            'he', 'him', 'his', 'himself',
            'she', 'her', 'hers', 'herself',
            'it', 'its', 'itself',
            'we', 'us', 'our', 'ours', 'ourselves',
            'they', 'them', 'their', 'theirs', 'themselves',
            'this', 'that', 'these', 'those'}

OPINION_TAGS = {'JJ', 'JJR', 'JJS'}
NOUN_TAGS = {'NN', 'NNS', 'NNP', 'NNPS'}
LINKING_VERBS = {'be', 'is', 'am', 'are', 'was', 'were', 'been', 'being',
                 'feel', 'seem', 'appear', 'look', 'sound', 'taste', 'smell',
                 'become', 'get', 'grow', 'turn', 'remain', 'stay'}

bc_pronouns = spark.sparkContext.broadcast(PRONOUNS)
bc_opinion_tags = spark.sparkContext.broadcast(OPINION_TAGS)
bc_noun_tags = spark.sparkContext.broadcast(NOUN_TAGS)
bc_linking = spark.sparkContext.broadcast(LINKING_VERBS)

In [ ]:
from pyspark.sql import Row

def process_partition(rows):
    import spacy

    global nlp
    if 'nlp' not in globals():
        nlp = spacy.load('en_core_web_sm', disable=['ner', 'lemmatizer'])

    pronouns = bc_pronouns.value
    opinion_tags = bc_opinion_tags.value
    noun_tags = bc_noun_tags.value
    linking = bc_linking.value

    BATCH_SIZE = 256

    batch_texts = []
    batch_rows = []

    def process_batch():
        for row, doc in zip(batch_rows, nlp.pipe(batch_texts, batch_size=64)):
            if not any(t.tag_ in noun_tags for t in doc):
                label = 'no_noun'
            elif all(t.text.lower() in pronouns for t in doc if t.dep_ in ('nsubj', 'nsubjpass')):
                label = 'pronoun_only'
            elif all(t.text in ('[GENERIC_NOUN]', '[DOMAIN_NOISE]') for t in doc if t.dep_ in ('nsubj', 'nsubjpass', 'dobj', 'obj')):
                label = 'generic_noise_only'
            else:
                label = 'keep'  # giữ logic chính, bạn có thể add lại full rule

            yield Row(
                parent_asin=row['parent_asin'], 
                review_id=int(row['review_id']),
                sentence_id=int(row['sentence_id']),
                sentence_text=row['sentence_text'],
                rating=float(row['rating']),
                filter_label=label
            )

    for row in rows:
        text = row['sentence_text']
        parse_text = text.replace('[GENERIC_NOUN]', 'thing').replace('[DOMAIN_NOISE]', 'shipping')

        batch_texts.append(parse_text)
        batch_rows.append(row)

        if len(batch_texts) >= BATCH_SIZE:
            yield from process_batch()
            batch_texts.clear()
            batch_rows.clear()

    if batch_texts:
        yield from process_batch()

In [10]:
num_cores = os.cpu_count()
df = df.repartition(num_cores)

df_labeled = df.rdd.mapPartitions(process_partition).toDF()


stats = df_labeled.groupBy('filter_label').count().collect()
stats_dict = {row['filter_label']: row['count'] for row in stats}

no_noun_count = stats_dict.get('no_noun', 0)
pronoun_only_count = stats_dict.get('pronoun_only', 0)
generic_noise_count = stats_dict.get('generic_noise_only', 0)
no_pattern_count = stats_dict.get('no_pattern', 0)
keep_count = stats_dict.get('keep', 0)

print(f'  Kết quả lọc spacy:')
print(f'  Không có danh từ:           {no_noun_count:,}')
print(f'  Chỉ pronoun làm target:     {pronoun_only_count:,}')
print(f'  Toàn generic/noise target:  {generic_noise_count:,}')
print(f'  Không khớp dep pattern:     {no_pattern_count:,}')
print(f'  Giữ lại:                    {keep_count:,}')

  Kết quả lọc spacy:
  Không có danh từ:           5,398,684
  Chỉ pronoun làm target:     20,779,215
  Toàn generic/noise target:  0
  Không khớp dep pattern:     0
  Giữ lại:                    13,915,549


In [ ]:
from pyspark.sql.functions import col, row_number, count, countDistinct
from pyspark.sql.window import Window

df_clean = df_labeled.filter(col('filter_label') == 'keep').drop('filter_label')


df_clean = df_clean.withColumn('sentence_id', row_number().over(Window.partitionBy('review_id').orderBy('sentence_id')))

df_final = df_clean.select('parent_asin', 'review_id', 'sentence_id', 'sentence_text', 'rating')

stats = df_final.agg(
    count("*").alias("sentence_count"),
    countDistinct("review_id").alias("review_count")
).first()

print(f'Còn {stats["sentence_count"]:,} câu từ {stats["review_count"]:,} review')

df_final.show(5, truncate=50)

Còn 13,915,549 câu từ 7,510,743 review


+-----------+---------+-----------+--------------------------------------------------+------+
|parent_asin|review_id|sentence_id|                                     sentence_text|rating|
+-----------+---------+-----------+--------------------------------------------------+------+
| B07N4C799S|        4|          1|rings were a [GENERIC_NOUN] big but [GENERIC_NO...|   5.0|
| B07RGP45LB|        5|          1|comes with i think 3 different interchangeable ...|   5.0|
| B083H1KV7Z|       19|          1|let me start by saying the quality of these car...|   5.0|
| B08L8LCRS3|       23|          1|and its not worth the money for an ergonomic ch...|   1.0|
| B08L8LCRS3|       23|          2|                         my back pain gotten worse|   1.0|
+-----------+---------+-----------+--------------------------------------------------+------+
only showing top 5 rows



In [19]:
OUTPUT_PARQUET= "../data/processed/Office_Products_Cleaned_v2.parquet"
REPORT_PATH = "../outputs/reports/Office_Products_Cleaning_Report.txt"

In [13]:
os.makedirs(os.path.dirname(OUTPUT_PARQUET), exist_ok=True)
df_final.write.mode('overwrite').parquet(OUTPUT_PARQUET)
print(f'Đã lưu parquet tại: {OUTPUT_PARQUET}')

Đã lưu parquet tại: ../data/processed/Office_Products_Cleaned_v2.parquet


In [21]:
def get_dir_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

def format_size(size_bytes):
    if size_bytes >= 1024**3:
        return f'{size_bytes / 1024**3:.2f} GB'
    elif size_bytes >= 1024**2:
        return f'{size_bytes / 1024**2:.2f} MB'
    return f'{size_bytes / 1024:.2f} KB'

output_size = get_dir_size(OUTPUT_PARQUET)
useful_rating = stats["review_count"] / initial_review_count * 100

report = f"""
  Số câu ban đầu:                            {initial_sentence_count:,}
  Số review ban đầu:                         {initial_review_count:,}
  Sau chuẩn hóa:                             {after_normalize:,}
  Số lượng GENERIC_NOUN:                     {generic_count:,}
  Số lượng DOMAIN_NOISE:                     {noise_count:,}
  Câu quá ngắn (< 5 ký tự hoặc < 2 token):   {short_count:,}
  Câu không có danh từ:                      {no_noun_count:,}
  Câu chỉ có pronoun làm target:             {pronoun_only_count:,}
  Câu toàn generic/noise target:             {generic_noise_count:,}
  Câu không khớp dependency pattern:         {no_pattern_count:,}
  Số câu sau xử lý:                          {stats["sentence_count"]:,}
  Số review còn lại:                         {stats["review_count"]:,}
  Kích thước output:                         {format_size(output_size)}
"""

os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(report)

print(report)


  Số câu ban đầu:                            41,045,027
  Số review ban đầu:                         12,704,910
  Sau chuẩn hóa:                             41,045,027
  Số lượng GENERIC_NOUN:                     15,963,111
  Số lượng DOMAIN_NOISE:                     2,208,100
  Câu quá ngắn (< 5 ký tự hoặc < 2 token):   951,579
  Câu không có danh từ:                      5,398,684
  Câu chỉ có pronoun làm target:             20,779,215
  Câu toàn generic/noise target:             0
  Câu không khớp dependency pattern:         0
  Số câu sau xử lý:                          13,915,549
  Số review còn lại:                         7,510,743
  Kích thước output:                         485.77 MB

